# 05 — OCR Experiments

**FinRAG learning series · Notebook 5 of 14**

## Learning objectives
1. Recognise scanned (image-only) PDF pages and understand why they need OCR.
2. Run **Tesseract** OCR through FinRAG's `ocr_service` (PyMuPDF + Tesseract).
3. Measure OCR accuracy on financial amounts against a ground truth, and see how resolution and preprocessing affect it.
4. Identify OCR errors that matter in finance (lost commas/parentheses, 1↔7, 0↔O) and how FinRAG flags OCR'd values.

## Prerequisites
Notebook 02. Tesseract language data installed (`tesseract-ocr`, `tesseract-ocr-eng`). Saved outputs use one statement page of a
public annual report.

## Concept
A scanned page is a picture: there is no text layer to extract. OCR (optical character recognition) segments the image into lines
and words and classifies each glyph. Its output is **probabilistic** — so FinRAG:
* only OCRs pages with almost no text and at least one image (`ocr_service.needs_ocr`);
* flags every OCR'd element (`Location.ocr = True`);
* lowers the confidence of OCR'd financial values and sets them to `needs_review`.

### Experiment design (honest ground truth)
We do not have a scanned report with a known transcription, so we **create** a scan from a digital page: render the page to an
image, wrap it in an image-only PDF, OCR it, and compare the amounts OCR finds with the amounts in the page's original text layer.
This measures OCR quality on genuine financial typography under controlled resolution. Real scans add skew, noise and blur, so
real-world accuracy is usually *lower* than what you will see here.

In [1]:
# Setup: make the production backend importable and keep secrets out of outputs
import sys, os
sys.path.insert(0, os.path.abspath("."))      # run notebooks from the notebooks/ folder
import finrag_nb as nb                          # helper: paths, safe settings, DB session
print("Backend package:", nb.BACKEND_DIR)
print("Documents folder:", nb.DOCS_DIR)

Backend package: /home/user/Ready_To_Deploy/backend
Documents folder: /opt/finrag_testdocs


In [2]:
from app.document_processing import ocr_service
print("Tesseract language data:", ocr_service.tessdata_dir())
print("OCR available to FinRAG:", ocr_service.available())
print("settings:", nb.safe_settings("ocr_enabled", "ocr_language", "ocr_min_chars"))

Tesseract language data: /usr/share/tesseract-ocr/5/tessdata
OCR available to FinRAG: True
settings: {'ocr_enabled': True, 'ocr_language': 'eng', 'ocr_min_chars': 25}


## 1. Ground truth from the text layer

In [3]:
import re, collections, pymupdf
from app.financial_analytics.numbers import is_number_token, looks_like_year

src = pymupdf.open(nb.doc_path("RIL-Integrated-Annual-Report-2024-25.pdf"))
PAGE = 100                      # consolidated statement of profit and loss in the saved run
page = src[PAGE - 1]

def amounts(words):
    # amount-like tokens with a thousands separator or decimals (excludes note numbers like '25')
    toks = [w[4].strip() for w in words]
    return collections.Counter(t for t in toks if is_number_token(t) and not looks_like_year(t) and ("," in t or "." in t))

truth = amounts(page.get_text("words"))
print(f"amount tokens in the text layer of page {PAGE}: {sum(truth.values())} ({len(truth)} distinct)")
print("examples:", list(truth)[:10])

amount tokens in the text layer of page 100: 67 (63 distinct)
examples: ['9,60,355', '8,83,646', '1,10,819', '1,16,476', '10,71,174', '10,00,122', '91,038', '85,650', '9,80,136', '9,14,472']


## 2. Simulate a scan and check that FinRAG would OCR it

In [4]:
def image_only_pdf(page, dpi, preprocess=None):
    pix = page.get_pixmap(dpi=dpi)
    if preprocess:
        pix = preprocess(pix)
    out = pymupdf.open()
    p = out.new_page(width=page.rect.width, height=page.rect.height)
    p.insert_image(p.rect, pixmap=pix)
    return out

scan = image_only_pdf(page, 150)
print("text characters on the simulated scan:", len(scan[0].get_text()))
print("needs OCR?", ocr_service.needs_ocr(scan[0]))

text characters on the simulated scan: 0
needs OCR? True


## 3. OCR at different resolutions

In [5]:
import time

def score(found, truth):
    tp = sum((found & truth).values())
    precision = tp / max(sum(found.values()), 1)
    recall = tp / max(sum(truth.values()), 1)
    return tp, precision, recall

results = {}
for dpi in (72, 150, 300):
    doc = image_only_pdf(page, dpi)
    t = time.time()
    words = ocr_service.ocr_words(doc[0], dpi=dpi)
    secs = time.time() - t
    found = amounts(words)
    tp, p, r = score(found, truth)
    results[dpi] = (found, words)
    print(f"dpi {dpi:>3}: {secs:5.1f}s | amounts found {sum(found.values()):>3} | correct {tp:>3} | precision {p:.1%} | recall {r:.1%}")

dpi  72:   0.2s | amounts found   0 | correct   0 | precision 0.0% | recall 0.0%


dpi 150:   2.9s | amounts found  66 | correct  64 | precision 97.0% | recall 95.5%


dpi 300:   4.6s | amounts found  66 | correct  66 | precision 100.0% | recall 98.5%


**Reading the table:** *recall* = share of the true amounts that OCR reproduced exactly (character for character, including commas
and parentheses); *precision* = share of the amounts OCR produced that are exactly right. Exact match is the right criterion for
finance: `9,80,136` read as `9,80,186` is simply wrong.

## 4. What kinds of errors does OCR make?

In [6]:
found150 = results[150][0]
missed = truth - found150
spurious = found150 - truth
print("true amounts OCR missed or misread (150 dpi):", sorted(missed.elements())[:15])
print("OCR outputs that are not in the text layer:", sorted(spurious.elements())[:15])

true amounts OCR missed or misread (150 dpi): ['1,16,476', '1,89,881', '51.47']
OCR outputs that are not in the text layer: ['116,476', '189,881']


In [7]:
# Pair each spurious token with the most similar true token to see the character-level error
import difflib
truth_list = sorted(truth)
for tok in sorted(spurious)[:10]:
    best = difflib.get_close_matches(tok, truth_list, n=1, cutoff=0.6)
    print(f"OCR {tok!r:>12}  <- probably {best[0]!r}" if best else f"OCR {tok!r:>12}  (no close match)")

OCR    '116,476'  <- probably '1,16,476'
OCR    '189,881'  <- probably '1,89,881'


## 5. Image preprocessing
Common preprocessing steps are grayscale conversion and binarisation (thresholding). We test grayscale at 150 dpi. Whether it helps
depends on the source; with clean renderings the gain is usually small — measure, don't assume.

In [8]:
def grayscale(pix):
    return pymupdf.Pixmap(pymupdf.csGRAY, pix)

doc = image_only_pdf(page, 150, preprocess=grayscale)
found_g = amounts(ocr_service.ocr_words(doc[0], dpi=150))
tp, p, r = score(found_g, truth)
print(f"grayscale 150 dpi: correct {tp} | precision {p:.1%} | recall {r:.1%}")

grayscale 150 dpi: correct 61 | precision 92.4% | recall 91.0%


## 6. The full parser on a scanned page

In [9]:
import tempfile
from app.document_processing.parsers.pdf_parser import PdfParser
from app.financial_analytics.metric_extractor import extract_metrics

tmp = tempfile.NamedTemporaryFile(suffix=".pdf", delete=False).name
image_only_pdf(page, 300).save(tmp)
parsed = PdfParser().parse(tmp)
print("OCR pages:", parsed.ocr_pages, "| warnings:", parsed.warnings)
ms = extract_metrics(parsed.elements, "pdf", None, None)
for m in [m for m in ms if m.metric_key in ("revenue_from_operations", "net_profit", "total_income")][:6]:
    print(f"{m.metric_key:<24} {m.raw_value:>10} {m.period_label!s:<10} confidence={m.confidence} ocr={m.location.ocr} notes={m.notes}")

OCR pages: [1] | warnings: []
revenue_from_operations    9,80,136 2024-25    confidence=0.65 ocr=True notes=['value read by OCR']
revenue_from_operations    9,14,472 2023-24    confidence=0.65 ocr=True notes=['value read by OCR']
total_income               9,98,114 2024-25    confidence=0.65 ocr=True notes=['value read by OCR']
total_income               9,30,529 2023-24    confidence=0.65 ocr=True notes=['value read by OCR']
net_profit                   81,309 2024-25    confidence=0.65 ocr=True notes=['value read by OCR']
net_profit                   79,020 2023-24    confidence=0.65 ocr=True notes=['value read by OCR']


## Interpretation
* **Resolution dominates.** In the saved run, 72 dpi recovered no amounts at all, 150 dpi recovered 64 of 67 exactly
  (recall 95.5%, precision 97.0%), and 300 dpi 66 of 67 (recall 98.5%, precision 100%). FinRAG's `ocr_words` default is 200 dpi
  as a speed/accuracy compromise.
* **The errors are instructive.** Both 150-dpi mistakes dropped a comma from Indian digit grouping (`1,16,476` → `116,476`): the
  digits survived, so `parse_number` would still give the right value, but an exact-text citation check would fail. The EPS value
  `51.47` was missed entirely. A missing or merged digit is the dangerous case, which is why OCR'd values are flagged.
* **Preprocessing is not automatically better.** Grayscale conversion *reduced* accuracy here (91.0% recall) on a clean rendering.
  Measure every preprocessing step on your own scans instead of assuming it helps.
* OCR'd values enter FinRAG with `ocr=True`, a 0.25 lower confidence (0.65 instead of 0.9) and a note; validation then marks them
  `needs_review`. Accounting identities (notebook 07) are a second line of defence because a misread digit usually breaks a sum.
* Caveat: the "scans" here are clean renderings of a digital page. Real scans with skew, noise and blur will score lower.

## Common errors and troubleshooting
| Symptom | Cause | Fix |
|---|---|---|
| "OCR (Tesseract) is not available" warning | No language data | `apt-get install tesseract-ocr tesseract-ocr-eng`, or set `TESSDATA_PREFIX` |
| Very low recall | Low-resolution scan | Rescan at ≥300 dpi |
| Columns merged | Skewed scan | Deskew before uploading |
| Non-English text garbled | Wrong language | Set `OCR_LANGUAGE` (e.g. `eng+hin`) and install that language pack |

## Practical exercises
1. Repeat the resolution experiment on another page (a dense notes page). Is recall lower?
2. Add a binarisation step (threshold the grayscale pixmap) and measure again.
3. Scan a printed page of your own (phone photo → PDF) and upload it to FinRAG. Which values does validation flag?

## Summary
You simulated scans of a genuine statement page, measured exact-match OCR accuracy on financial amounts at several resolutions,
inspected real OCR errors, and saw how FinRAG flags OCR output.

## Further learning
* Tesseract documentation: https://tesseract-ocr.github.io/
* PyMuPDF OCR: https://pymupdf.readthedocs.io/en/latest/recipes-ocr.html
* FinRAG source: `backend/app/document_processing/ocr_service.py`
* Next: **06 — Financial data extraction**.